# LAMU 2026 - mapa nowych pytan na modelu wytrenowanym na 2025

### Co robi ten notatnik?

1. Bierze pytania **LAMU 2025** i koduje je modelem **`OPI-PIB/PolDense-1B`** (ModernBERT, kolekcja PolDense/EuroDense OPI-PIB).
2. **Trenuje UMAP na pytaniach z 2025** - to jest "mapa bazowa", ustalona raz i niezmienna.
3. Dokleja **nowe pytania z LAMU 2026** i **projektuje je na te sama mape** (`reducer.transform()`, a NIE `fit_transform`). Nowe punkty laduja w tej samej przestrzeni co pytania z 2025, wiec widac, obok ktorych tematow sie znajduja.

Nowe pytania 2026 edytujesz w jednym miejscu (slownik `new_questions_2026` w sekcji 4) - dokladasz kolejne w miare jak rosnie plan LAMU 2026.

### Jak uruchomic

1. [Google Colab](https://colab.research.google.com/) -> wgraj ten plik (.ipynb).
2. **Srodowisko uruchomieniowe > Zmien typ srodowiska > GPU (T4)** - PolDense-1B ma ~1 mld parametrow, na GPU jest znacznie szybszy.
3. **Uruchom wszystko** (Ctrl+F9). Pierwsze pobranie modelu (~2 GB) trwa kilka minut.

In [ ]:
!pip install -q -U "sentence-transformers>=5.4.0" transformers umap-learn plotly pymupdf

In [ ]:
import fitz  # pymupdf
import re
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
import umap
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = 'colab'

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Urzadzenie: {DEVICE}")


def emb_dim(model):
    # get_sentence_embedding_dimension() zostalo przemianowane na get_embedding_dimension()
    try:
        return model.get_embedding_dimension()
    except AttributeError:
        return model.get_sentence_embedding_dimension()

## 1. Pobranie i wczytanie pytan z 2025 (PDF)

To jest zbior, na ktorym trenujemy mape. (Ten sam parser co w pozostalych notatnikach.)

In [ ]:
import requests

PDF_URL = "https://radionaukowe.pl/wp-content/uploads/2026/03/LAMU-zadane-pytania-2026.pdf"
PDF_PATH = "LAMU-zadane-pytania-2026.pdf"

response = requests.get(PDF_URL)
response.raise_for_status()
with open(PDF_PATH, "wb") as f:
    f.write(response.content)
print(f"Pobrano PDF: {PDF_PATH} ({len(response.content) / 1024:.0f} KB)")

In [ ]:
def extract_questions_from_pdf(pdf_path):
    """Extract questions grouped by topic from the LAMU PDF."""
    doc = fitz.open(pdf_path)
    full_text = ""
    for page in doc:
        full_text += page.get_text()
    doc.close()

    topic_pattern = r'(FIZYKA|BIOLOGIA|WSZECH[ŚS]WIAT|CZ[ŁL]OWIEK|TECHNOLOGIE|ZIEMIA|MATEMATYKA|HISTORIA|CHEMIA)'
    parts = re.split(topic_pattern, full_text)

    questions = []
    current_topic = None

    for part in parts:
        part_stripped = part.strip()
        if re.match(topic_pattern, part_stripped):
            current_topic = part_stripped.replace('Ś', 'S').replace('Ł', 'L')
            topic_display = {
                'FIZYKA': 'Fizyka', 'BIOLOGIA': 'Biologia', 'WSZECHSWIAT': 'Wszechświat',
                'CZLOWIEK': 'Człowiek', 'TECHNOLOGIE': 'Technologie', 'ZIEMIA': 'Ziemia',
                'MATEMATYKA': 'Matematyka', 'HISTORIA': 'Historia', 'CHEMIA': 'Chemia'
            }
            current_topic = topic_display.get(current_topic, current_topic)
            continue

        if current_topic is None:
            continue

        lines = part.split('\n')
        buffer = ""
        for line in lines:
            line = line.strip()
            if not line or line.startswith('LAMU') or line.startswith('Radio') or 'RADIO' in line.upper() or 'NAUKOWE' in line.upper() or 'LETNIA' in line.upper() or 'AKADEMIA' in line.upper():
                continue
            if re.match(r'^[–\-]\s', line):
                if buffer:
                    questions.append((current_topic, buffer.strip()))
                buffer = re.sub(r'^[–\-]\s*', '', line)
            else:
                if buffer:
                    buffer += ' ' + line

        if buffer:
            questions.append((current_topic, buffer.strip()))

    return questions

questions_data = extract_questions_from_pdf(PDF_PATH)
df = pd.DataFrame(questions_data, columns=['topic', 'question'])
print(f"Pytania 2025: {len(df)}")
print(df['topic'].value_counts())
df.head(5)

## 2. Model embeddingow (PolDense-1B)

PolDense-1B (ModernBERT) ladujemy z `bfloat16` i fallbackiem implementacji uwagi: `flash_attention_2` -> `sdpa` -> `eager` (na Colab T4 zwykle konczy sie na `sdpa`, i to w pelni wystarcza).

Modele PolDense wymagaja prefiksu tekstu - dla mapy podobienstwa uzywamy `[sts]: `.

> Aby wrocic do starego modelu, ustaw `MODEL_ID = "sdadas/st-polish-paraphrase-from-distilroberta"` i `PREFIX = ""`.

In [ ]:
MODEL_ID = "OPI-PIB/PolDense-1B"
PREFIX = "[sts]: "   # dla starego modelu ustaw ""

def load_model(model_id, device):
    if device == "cuda" and model_id.startswith("OPI-PIB/"):
        for attn in ["flash_attention_2", "sdpa", "eager"]:
            try:
                m = SentenceTransformer(model_id, device=device,
                    model_kwargs={"dtype": "bfloat16", "attn_implementation": attn})
                print(f"  {model_id}: zaladowano (attn={attn}, bfloat16)")
                return m
            except Exception as e:
                print(f"  {model_id}: attn={attn} nie zadzialalo -> {type(e).__name__}")
    m = SentenceTransformer(model_id, device=device)
    print(f"  {model_id}: zaladowano (domyslne ustawienia, {device})")
    return m

model = load_model(MODEL_ID, DEVICE)
print(f"  wymiar embeddingow: {emb_dim(model)}")


def encode(texts):
    return model.encode([PREFIX + t for t in texts], show_progress_bar=True,
                        batch_size=16, convert_to_numpy=True,
                        normalize_embeddings=True).astype(np.float32)

## 3. Embedding + trening UMAP na pytaniach 2025

Trenujemy UMAP raz, na pytaniach 2025 (`fit_transform`). Zapisujemy `reducer`, zeby pozniej wprojektowac na niego pytania 2026.

In [ ]:
emb_2025 = encode(df['question'].tolist())
print(f"Embeddingi 2025: {emb_2025.shape}")

reducer = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1,
                    metric='cosine', random_state=42)
coords = reducer.fit_transform(emb_2025)
df['x'] = coords[:, 0]
df['y'] = coords[:, 1]
print(f"Wytrenowano UMAP na {len(df)} pytaniach 2025: {emb_2025.shape[1]} -> 2")

## 4. Nowe pytania LAMU 2026 (edytuj tutaj)

Kazda sekcja to lista pytan. Dokladaj kolejne w miare rozwoju planu 2026. Nazwiska/wiek dzieci oraz linie "Odpowiada prof. ..." pomijamy - zostaje samo pytanie.

In [ ]:
new_questions_2026 = {
    "Bakterie": [
        "Co było pierwsze, bakterie czy wirusy?",
        "Skąd bakterie wiedzą, co mają robić?",
        "Czy we łzach są bakterie?",
        "Skoro antybiotyki zabijają bakterie, to czemu nie potrafią zabić wirusów?",
        "Skoro bakterie są, a ich nie widać, to może krasnoludki też są?",
        "Jaka jest najmniejsza komórka na świecie?",
    ],
    "Gnicie, pleśnienie i kupa": [
        "Dlaczego owoce i warzywa gniją?",
        "Jak jedzenie gnije?",
        "Dlaczego niektóre rzeczy pleśnieją, a inne nie?",
        "Dlaczego rzeczy w lodówce pleśnieją wolniej?",
        "Czemu jabłko staje się brązowe po miesiącu, dwóch w chłodnej temperaturze?",
        "Dlaczemu kupy psa zmieniają kolor po jakimś czasie na biały?",
    ],
    "Muchy i kowale": [
        "Czy muchy mają tak jak my organy wewnętrzne, na przykład jak wątroba, mózg i serce?",
        "Z czego się składa organizm muchy?",
        "Dlaczego mucha lata w tak chaotyczny sposób? Czy ma wyznaczoną jakąś trajektorię?",
        "Czemu muchy muszą mieć krew, żeby złożyć jaja?",
        "Dlaczego kowale bezskrzydłe się łączą?",
    ],
}

df_new = pd.DataFrame(
    [(sec, q) for sec, qs in new_questions_2026.items() for q in qs],
    columns=['section', 'question'])
print(f"Nowych pytań 2026: {len(df_new)} (sekcje: {df_new['section'].nunique()})")
df_new

## 5. Projekcja pytan 2026 na wytrenowana mape

`reducer.transform()` uzywa juz wytrenowanego UMAP - nie zmienia pozycji pytan 2025, tylko znajduje miejsce dla nowych.

In [ ]:
emb_new = encode(df_new['question'].tolist())
new_coords = reducer.transform(emb_new)
df_new['x'] = new_coords[:, 0]
df_new['y'] = new_coords[:, 1]
print(f"Wprojektowano {len(df_new)} pytań 2026 na mapę 2025")
df_new[['section', 'question', 'x', 'y']]

## 6. Interaktywna mapa: 2025 (kategorie) + 2026 (gwiazdki)

Punkty 2025 sa przygaszone i pokolorowane wg kategorii z PDF. Pytania 2026 to duze gwiazdki pokolorowane wg sekcji planu. Najedz myszka, by zobaczyc tresc; kliknij pozycje w legendzie, by ja ukryc/wyizolowac.

In [ ]:
topic_colors = {
    'Fizyka': '#1f77b4', 'Biologia': '#2ca02c', 'Wszechświat': '#9467bd',
    'Człowiek': '#d62728', 'Technologie': '#ff7f0e', 'Ziemia': '#8c564b',
    'Matematyka': '#e377c2', 'Historia': '#7f7f7f', 'Chemia': '#17becf'
}
section_colors = {'Bakterie': '#e6194B', 'Gnicie, pleśnienie i kupa': '#3cb44b',
                  'Muchy i kowale': '#4363d8'}
extra = ['#f58231', '#911eb4', '#42d4f4', '#bfef45', '#f032e6']

def wrap(q):
    return '<br>'.join(q[i:i+70] for i in range(0, len(q), 70))

df['q_hover'] = df['question'].apply(wrap)
df_new['q_hover'] = df_new['question'].apply(wrap)

fig = go.Figure()
# tlo: pytania 2025 wg kategorii
for topic in df['topic'].value_counts().index:
    sub = df[df['topic'] == topic]
    fig.add_trace(go.Scatter(
        x=sub['x'], y=sub['y'], mode='markers', name=topic, legendgroup=topic,
        marker=dict(size=7, color=topic_colors.get(topic, '#333333'),
                    opacity=0.35, line=dict(width=0)),
        customdata=sub[['topic', 'q_hover']].values,
        hovertemplate='<b>2025 · %{customdata[0]}</b><br>%{customdata[1]}<extra></extra>'))
# nowe pytania 2026 wg sekcji
for i, sec in enumerate(new_questions_2026):
    sub = df_new[df_new['section'] == sec]
    fig.add_trace(go.Scatter(
        x=sub['x'], y=sub['y'], mode='markers', name=f'2026: {sec}',
        legendgroup=f'2026: {sec}',
        marker=dict(size=16, symbol='star',
                    color=section_colors.get(sec, extra[i % len(extra)]),
                    opacity=1.0, line=dict(width=1.2, color='black')),
        customdata=sub[['section', 'q_hover']].values,
        hovertemplate='<b>2026 · %{customdata[0]}</b><br>%{customdata[1]}<extra></extra>'))

fig.update_layout(
    title=f'LAMU 2026 - nowe pytania (gwiazdki) na mapie 2025 (model: {MODEL_ID})',
    legend_title_text='Legenda', font=dict(size=12), width=1000, height=750,
    plot_bgcolor='white', hoverlabel=dict(font_size=11),
    xaxis=dict(title='UMAP 1', showgrid=True, gridcolor='lightgray'),
    yaxis=dict(title='UMAP 2', showgrid=True, gridcolor='lightgray'))
fig.show()

HTML_PATH = "lamu_2026_map.html"
fig.write_html(HTML_PATH, include_plotlyjs=True, full_html=True)
print(f"Zapisano: {HTML_PATH}")

## 7. Statyczny wykres (PNG)

In [ ]:
fig2, ax = plt.subplots(figsize=(13, 10))
for topic in df['topic'].unique():
    m = df['topic'] == topic
    ax.scatter(df.loc[m, 'x'], df.loc[m, 'y'], color=topic_colors.get(topic, '#333333'),
               s=40, alpha=0.30, edgecolors='none', label=topic)
for i, sec in enumerate(new_questions_2026):
    m = df_new['section'] == sec
    ax.scatter(df_new.loc[m, 'x'], df_new.loc[m, 'y'], marker='*', s=280,
               color=section_colors.get(sec, extra[i % len(extra)]),
               edgecolors='black', linewidth=0.8, label=f'2026: {sec}')
ax.legend(fontsize=9, loc='best', framealpha=0.9)
ax.set_title(f'LAMU 2026 na mapie 2025 (model: {MODEL_ID})')
ax.set_xlabel('UMAP 1'); ax.set_ylabel('UMAP 2'); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('lamu_2026_static.png', dpi=150, bbox_inches='tight')
plt.show()
print("Zapisano: lamu_2026_static.png")

## 8. Eksport plikow

In [ ]:
for path in ["lamu_2026_map.html", "lamu_2026_static.png"]:
    try:
        from google.colab import files
        files.download(path)
    except ImportError:
        print(f"Otworz lokalnie: {path}")